In [26]:
import os
import pandas as pd
import gseapy as gp
import json
from pathlib import Path

## ORA

In [27]:
organism='Mouse'

In [ ]:
#contrast = "YourContrastOfInterest"
#df = pd.read_csv(f"./results/contrasts/{contrast}.csv",sep=';')
#df

,gene_id,gene_name,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj
0,ENSMUSG00000000001,Gnai3,6614.435806,0.075209,0.061264,1.227622,0.219589,0.586436
1,ENSMUSG00000000028,Cdc45,2677.765983,-0.051781,0.067851,-0.763154,0.445371,0.787654
2,ENSMUSG00000000031,H19,3.545939,0.109256,0.764521,0.142908,0.886363,0.973293
3,ENSMUSG00000000037,Scml2,506.453130,-0.380692,0.134451,-2.831449,0.004634,0.059218
4,ENSMUSG00000000056,Narf,2833.805782,0.078905,0.161024,0.490018,0.624121,0.887025
...,...,...,...,...,...,...,...,...
22284,ENSMUSG00002076556,Gm56424,16.984368,-0.499397,0.990155,-0.504363,0.614007,0.882776
22285,ENSMUSG00002076601,Scarna13,0.616981,0.053467,2.036357,0.026256,0.979053,NaN
22286,ENSMUSG00002076665,Gm54427,0.485802,-0.620040,2.841005,-0.218247,0.827237,NaN
22287,ENSMUSG00002076675,Gm56334,0.729818,-1.141471,2.069091,-0.551678,0.581169,NaN


In [29]:
def compute_ORA(df,database_name='GO_Biological_Process_2025',organism='mouse'):
    gene_list = df[df['padj']<0.05]['gene_name'].tolist()
    # if you are only intrested in dataframe that enrichr returned, please set outdir=None
    enr = gp.enrichr(gene_list=gene_list, # or "./tests/data/gene_list.txt",
                    gene_sets=[database_name],
                    organism=organism, # don't forget to set organism to the one you desired! e.g. Yeast
                    outdir=None, # don't write to disk
                    )
    ora_results = enr.results
    ora_results['Overlap_Ratio'] = ora_results['Overlap'].apply(lambda x: int(x.split('/')[0]) / int(x.split('/')[1]))
    ora_results['Count'] = ora_results['Overlap'].apply(lambda x: int(x.split('/')[0]))
    return ora_results

In [30]:
def full_go_analysis(df,contrast,organism='mouse'):
    organism = organism.lower()
    ora_res = compute_ORA(df,organism=organism)
    ora_res.to_csv(f"./results/pathway/ORA/ORA_GO_{contrast}_ALL.csv",sep=';')
    
    ora_res_up = compute_ORA(df[df['log2FoldChange']>0],organism=organism)
    ora_res_up.to_csv(f"./results/pathway/ORA/ORA_GO_{contrast}_UP.csv",sep=';')
    
    ora_res_down = compute_ORA(df[df['log2FoldChange']<0],organism=organism)
    ora_res_down.to_csv(f"./results/pathway/ORA/ORA_GO_{contrast}_DOWN.csv",sep=';')

In [ ]:
# for testing ORA function
#full_go_analysis(df,contrast,organism=organism)

## GSEA

In [37]:
# Read data
with open(f'data/DB/{organism}/GO_Biological_Process_2025.json', 'r') as f:
    GO = json.load(f)

if organism == 'Mouse':
    with open(f'data/DB/{organism}/KEGG_2019_Mouse.json', 'r') as f:
        KEGG = json.load(f)
else:
    with open(f'data/DB/{organism}/KEGG_2026.json', 'r') as f:
        KEGG = json.load(f)

with open(f'data/DB/{organism}/MSigDB_Hallmark_2020.json', 'r') as f:
    HM = json.load(f)


In [ ]:
DBs = {'GO':GO,'KEGG':KEGG,'Hallmark':HM}

In [39]:
import numpy as np
import pandas as pd
import gseapy as gp

def compute_GSEA(df: pd.DataFrame, database: str) -> pd.DataFrame:
    """
    Compute preranked GSEA using gseapy with ranking = sign(log2FC) * -log10(pvalue).

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame containing at least 'gene_name', 'log2FoldChange', and 'pvalue'.
    database : str
        Gene set database path or name (GMT file or Enrichr library).

    Returns
    -------
    gsea_res : pd.DataFrame
        GSEA results with additional columns:
        - 'Tag_%' : fraction of genes in the gene set
        - '-log10Qvalue' : -log10(FDR q-value)
    """

    # Remove genes with missing p-values
    df_cleaned = df.dropna(subset=['pvalue']).copy()

    # Compute rank: sign(log2FC) * -log10(pvalue)
    df_cleaned['rank'] = df_cleaned.apply(
        lambda x: np.sign(x['log2FoldChange']) * -np.log10(x['pvalue']),
        axis=1
    )

    # Prepare prerank input: unique gene names, uppercase
    rnk = (
        df_cleaned[['gene_name', 'rank']]
        .sort_values(by='rank', ascending=False)
        .drop_duplicates(subset=['gene_name'], keep='first')
        .dropna()
    )
    rnk['gene_name'] = rnk['gene_name'].str.upper()

    # Run preranked GSEA
    pre_res = gp.prerank(
        rnk=rnk,
        gene_sets=database,
        min_size=10,
        max_size=500,
        permutation_num=1000,  # reduce for speed in testing
        outdir=None,            # do not write to disk
        seed=6,
        verbose=False,
        threads=12,
    )

    gsea_res = pre_res.res2d.copy()

    # Compute fraction of genes in the gene set (Tag %)
    tag_split = gsea_res['Tag %'].str.split("/", expand=True)
    gsea_res['Tag_%'] = tag_split[0].astype(int) / tag_split[1].astype(int)

    # Handle zero FDR values for -log10 transformation
    non_zero_fdr_min = gsea_res[gsea_res['FDR q-val'] != 0.0]['FDR q-val'].min()
    gsea_res['FDR q-val non zero'] = gsea_res['FDR q-val'].apply(
        lambda x: x if x != 0.0 else non_zero_fdr_min
    )
    gsea_res['-log10Qvalue'] = -np.log10(gsea_res['FDR q-val non zero'])

    return gsea_res


In [ ]:
def full_gsea_analysis(df,contrast,DBs):

    for DB_name in DBs:
        print(f"    - Database: {DB_name}")
        gsea_res = compute_GSEA(df,DBs[DB_name])
        gsea_res.to_csv(f"./results/pathway/GSEA/GSEA_{DB_name}_{contrast}.csv",sep=';')

In [ ]:
#full_gsea_analysis(df,contrast,DBs)

    DB: GO
    DB: KEGG
    DB: HM


In [42]:
path = "./results/contrasts/"

for file in os.listdir(path):
    if os.path.isfile(os.path.join(path, file)):
        df = pd.read_csv(f"{path}{file}",sep=';')
        contrast = Path(file).stem
        print("compute ORA for: ",contrast)
        full_go_analysis(df,contrast)
        print("compute GSEA for: ",contrast)
        full_gsea_analysis(df,contrast,DBs)
    else:
        print("this is not a file")

compute ORA for:  condition_Trained_Young_VS_Control_Young
compute GSEA for:  condition_Trained_Young_VS_Control_Young
    DB: GO
    DB: KEGG
    DB: HM
compute ORA for:  condition_Control_Old_VS_Control_Young
compute GSEA for:  condition_Control_Old_VS_Control_Young
    DB: GO
    DB: KEGG
    DB: HM
compute ORA for:  condition_Trained_Old_VS_Control_Old
compute GSEA for:  condition_Trained_Old_VS_Control_Old
    DB: GO
    DB: KEGG
    DB: HM
compute ORA for:  condition_Trained_Old_VS_Trained_Young
compute GSEA for:  condition_Trained_Old_VS_Trained_Young
    DB: GO
    DB: KEGG
    DB: HM


## Concat pathways

In [ ]:
dir_path_GSEA = './results/pathway/GSEA/'
dir_path_ORA = './results/pathway/ORA/'

In [ ]:
list_df_GSEA = []
for file in os.listdir(dir_path_GSEA):
    file_noExtension = file.split(".")[0]
    print(file_noExtension)
    split = file_noExtension.split("_")
    algo = split[0]
    db = split[1]
    deseq2 = split[2]
    contrast = "_".join(split[3:])
    df = pd.read_csv(dir_path_GSEA+file,delimiter=';',index_col=0)
    df['algo'] = algo
    df['db'] = db
    df['deseq2'] = deseq2
    df['contrast'] = contrast
    list_df_GSEA += [df]

GSEA_GO_condition_Control_Old_VS_Control_Young
GSEA_KEGG_condition_Control_Old_VS_Control_Young
GSEA_HM_condition_Control_Old_VS_Control_Young
GSEA_GO_condition_Trained_Young_VS_Control_Young
GSEA_KEGG_condition_Trained_Young_VS_Control_Young
GSEA_HM_condition_Trained_Young_VS_Control_Young
GSEA_GO_condition_Trained_Old_VS_Control_Old
GSEA_KEGG_condition_Trained_Old_VS_Control_Old
GSEA_HM_condition_Trained_Old_VS_Control_Old
GSEA_GO_condition_Trained_Old_VS_Trained_Young
GSEA_KEGG_condition_Trained_Old_VS_Trained_Young
GSEA_HM_condition_Trained_Old_VS_Trained_Young


In [ ]:
concat_GSEA = pd.concat(list_df_GSEA)

In [ ]:
concat_GSEA.head(3)

,Name,Term,ES,NES,NOM p-val,FDR q-val,FWER p-val,Tag %,Gene %,Lead_genes,Tag_%,FDR q-val non zero,-log10Qvalue,algo,db,deseq2,contrast
0,prerank,Protein Localization to Chromosome (GO:0034502),-0.674381,-2.127981,0.0,0.627838,0.240,17/46,14.09%,TNKS2;LEF1;CENPA;TERF2;TONSL;TERF2IP;BRD3;BRD2...,0.369565,0.627838,0.202152,GSEA,GO,condition,Control_Old_VS_Control_Young
1,prerank,Transcription Initiation at RNA Polymerase II ...,-0.586586,-2.081687,0.0,0.468305,0.293,38/101,19.12%,TAF4;MBD4;MED26;EP300;GTF2B;PHF2;RBBP5;ATAD2B;...,0.376238,0.468305,0.329471,GSEA,GO,condition,Control_Old_VS_Control_Young
2,prerank,Positive Regulation of Telomere Maintenance vi...,-0.778045,-2.028273,0.0,0.447721,0.361,1/20,0.04%,TNKS2,0.050000,0.447721,0.348993,GSEA,GO,condition,Control_Old_VS_Control_Young


In [ ]:
concat_GSEA.to_csv("./results/pathway/concat/concat_GSEA.csv")

In [ ]:
list_df_ORA = []
for file in os.listdir(dir_path_ORA):
    file_noExtension = file.split(".")[0]
    print(file_noExtension)
    split = file_noExtension.split("_")
    algo = split[0]
    db = split[1]
    deseq2 = split[2]
    contrast = "_".join(split[3:-1])
    direction = split[-1]
    df = pd.read_csv(dir_path_ORA+file,delimiter=';',index_col=0)
    df['algo'] = algo
    df['db'] = db
    df['deseq2'] = deseq2
    df['contrast'] = contrast
    df['direction']=direction
    list_df_ORA += [df]

ORA_GO_condition_Control_Old_VS_Control_Young_ALL
ORA_GO_condition_Control_Old_VS_Control_Young_UP
ORA_GO_condition_Control_Old_VS_Control_Young_DOWN
ORA_GO_condition_Trained_Young_VS_Control_Young_ALL
ORA_GO_condition_Trained_Young_VS_Control_Young_UP
ORA_GO_condition_Trained_Young_VS_Control_Young_DOWN
ORA_GO_condition_Trained_Old_VS_Control_Old_ALL
ORA_GO_condition_Trained_Old_VS_Control_Old_UP
ORA_GO_condition_Trained_Old_VS_Control_Old_DOWN
ORA_GO_condition_Trained_Old_VS_Trained_Young_ALL
ORA_GO_condition_Trained_Old_VS_Trained_Young_UP
ORA_GO_condition_Trained_Old_VS_Trained_Young_DOWN


In [ ]:
concat_ORA = pd.concat(list_df_ORA)

In [ ]:
concat_ORA.head(3)

,Gene_set,Term,Overlap,P-value,Adjusted P-value,Old P-value,Old Adjusted P-value,Odds Ratio,Combined Score,Genes,Overlap_Ratio,Count,algo,db,deseq2,contrast,direction
0,GO_Biological_Process_2025,Integrin-Mediated Signaling Pathway (GO:0007229),18/86,0.000010,0.022959,0,0,3.796429,43.716753,SEMA7A;COL16A1;ITGB5;DST;SRC;ITGB3;ITGA2B;PLEK...,0.209302,18,ORA,GO,condition,Control_Old_VS_Control_Young,ALL
1,GO_Biological_Process_2025,Cellular Response to Transforming Growth Facto...,19/98,0.000018,0.022959,0,0,3.449977,37.653776,JUN;ITGB5;HPGD;ZFYVE9;SRC;STAT3;LTBP3;PTPRK;FO...,0.193878,19,ORA,GO,condition,Control_Old_VS_Control_Young,ALL
2,GO_Biological_Process_2025,Regulation of Platelet Aggregation (GO:0090330),9/26,0.000025,0.022959,0,0,7.561231,80.049136,IL6;SERPINE2;MMRN1;CD9;EMILIN1;PRKCA;PRKCQ;F11...,0.346154,9,ORA,GO,condition,Control_Old_VS_Control_Young,ALL


In [ ]:
concat_ORA.to_csv("./results/pathway/concat/concat_ORA.csv")